# P4 — Generation TTS pour Audiobook

**Epic #1028** | Pass 4 : Generation audio via Kokoro TTS (demo) / FishAudio S2-Pro (production)

## Objectifs

1. Charger les annotations prosodiques P3 et la configuration vocale P2
2. Mapper chaque segment au bon modele/voix TTS
3. Generer les fichiers audio pour les 14 segments du texte
4. Exporter les metadonnees de generation (durees, tailles, paths)

In [1]:
import json
import os
import time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional
import requests

# Paths
BASE_DIR = Path(".")
PROSODIC_FILE = BASE_DIR / "prosodic_output" / "prosodic_annotations.json"
VOICE_CONFIG = BASE_DIR / "voice_casting_output" / "voice_casting_config.json"
TTS_OUTPUT_DIR = BASE_DIR / "tts_output"
TTS_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

KOKORO_URL = "http://localhost:8191"
KOKORO_API_KEY = os.getenv("KOKORO_API_KEY", "")
print(f"Output dir: {TTS_OUTPUT_DIR.resolve()}")
print(f"Prosodic annotations: {PROSODIC_FILE.exists()}")
print(f"Voice config: {VOICE_CONFIG.exists()}")
print(f"Kokoro API key: {'set' if KOKORO_API_KEY else 'NOT SET'}")

Output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\tts_output
Prosodic annotations: True
Voice config: True
Kokoro API key: set


## Architecture TTS

| Modele | Statut | Qualite | Latence | Expressivite |
|--------|--------|---------|---------|-------------|
| **Kokoro** (local Docker) | Actif | Bonne | <2s | Tags base (speed, emotion) |
| **FishAudio S2-Pro** | Non deploye | Excellente | ~3s | 15000+ tags expressifs |
| **OpenAI TTS-1** | Cloud | Bonne | ~2s | 6 voix fixes |

Ce notebook utilise **Kokoro** pour la demo locale. Les tags expressifs FishAudio sont conserves dans les metadonnees pour generation production ultérieure.

In [2]:
def load_voice_config(path):
    """Load P2 voice casting configuration."""
    with open(path) as f:
        return json.load(f)

def load_prosodic_annotations(path):
    """Load P3 prosodic annotations."""
    with open(path) as f:
        return json.load(f)

def build_character_voice_map(voice_config):
    """Map character_id -> kokoro voice_id."""
    mapping = {}
    for char_id, char_data in voice_config["characters"].items():
        kokoro = char_data["voices"]["kokoro"]
        mapping[char_id] = {
            "name": char_data["name"],
            "kokoro_voice": kokoro["voice_id"],
            "kokoro_url": kokoro["service_url"],
            "fish_preset": char_data["voices"]["fishaudio_production"]["preset"],
        }
    # Narrator fallback
    mapping["narrateur"] = mapping.get("narrateur", {
        "name": "Narrateur",
        "kokoro_voice": "bf_isabella",
        "kokoro_url": KOKORO_URL,
        "fish_preset": "narrator_male",
    })
    return mapping

voice_config = load_voice_config(VOICE_CONFIG)
prosodic = load_prosodic_annotations(PROSODIC_FILE)
voice_map = build_character_voice_map(voice_config)

print(f"Characters mapped: {len(voice_map)}")
for cid, info in voice_map.items():
    print(f"  {cid}: {info['name']} -> Kokoro {info['kokoro_voice']}")

Characters mapped: 8
  elisabeth_rousset: Elisabeth Rousset -> Kokoro af_sky
  cornudet: Cornudet -> Kokoro bm_george
  comtesse: Comtesse de Breville -> Kokoro af_sarah
  comte: Comte de Breville -> Kokoro am_michael
  narrateur: Narrateur -> Kokoro bf_isabella
  carre_lamadon: Monsieur Carre-Lamadon -> Kokoro bm_george
  loiseau: Monsieur Loiseau -> Kokoro bm_lewis
  soeurs: Les deux soeurs -> Kokoro bf_emma


## Kokoro TTS — API locale

Le service Kokoro (Docker, port 8191) expose une API compatible OpenAI :

```
POST /v1/audio/speech
Content-Type: application/json

{"model": "kokoro", "voice": "af_sky", "input": "texte", "speed": 1.0}
```

Reponse : audio MP3 (128 kbps, 24000 Hz, mono).

Note : Les tags expressifs `[whisper]`, `[shout]` etc. sont ignores par Kokoro mais conserves pour FishAudio S2-Pro en production.

In [3]:
@dataclass
class TTSRequest:
    """Represents a TTS generation request for one segment."""
    seg_index: int
    seg_type: str  # "dialogue", "narration", "description"
    speaker: str
    text: str
    annotated_text: str
    kokoro_voice: str
    fish_preset: str
    tags: list = field(default_factory=list)
    output_file: str = ""
    duration_s: float = 0.0
    file_size_kb: float = 0.0
    status: str = "pending"

def kokoro_tts(text, voice_id, speed=1.0):
    """Generate audio via Kokoro TTS API."""
    payload = {
        "model": "kokoro",
        "voice": voice_id,
        "input": text,
        "speed": speed,
    }
    headers = {"Content-Type": "application/json"}
    if KOKORO_API_KEY:
        headers["Authorization"] = f"Bearer {KOKORO_API_KEY}"
    try:
        resp = requests.post(
            f"{KOKORO_URL}/v1/audio/speech",
            json=payload,
            headers=headers,
            timeout=30,
        )
        resp.raise_for_status()
        return resp.content
    except requests.RequestException as e:
        print(f"  TTS error for voice={voice_id}: {e}")
        return None

print("TTS client ready")
print(f"Kokoro endpoint: {KOKORO_URL}")

TTS client ready
Kokoro endpoint: http://localhost:8191


## Construction des segments a generer

Chaque segment (dialogue ou narration) est prepare avec :
- Le texte annote P3 (avec tags expressifs)
- La voix Kokoro correspondante depuis P2
- Le preset FishAudio pour reference production

In [4]:
def resolve_speaker_id(speaker_name, voice_map):
    """Resolve a speaker name to character_id."""
    name_lower = speaker_name.lower().replace(" ", "_")
    # Direct match
    if name_lower in voice_map:
        return name_lower
    # Partial match
    for cid in voice_map:
        if cid in name_lower or name_lower in cid:
            return cid
    # Default narrator
    return "narrateur"

tts_requests = []

# Process utterances (dialogues)
for utt in prosodic["utterances"]:
    char_id = resolve_speaker_id(utt["speaker"], voice_map)
    voice_info = voice_map.get(char_id, voice_map["narrateur"])
    tts_requests.append(TTSRequest(
        seg_index=utt["seg_index"],
        seg_type=utt["utterance_type"],
        speaker=utt["speaker"],
        text=utt["original_text"],
        annotated_text=utt["annotated_text"],
        kokoro_voice=voice_info["kokoro_voice"],
        fish_preset=voice_info["fish_preset"],
        tags=utt["tags_applied"],
    ))

# Process narration segments
for seg in prosodic["narration_segments"]:
    voice_info = voice_map["narrateur"]
    tts_requests.append(TTSRequest(
        seg_index=seg["seg_index"],
        seg_type=seg["seg_type"],
        speaker="Narrateur",
        text=seg["original_text"],
        annotated_text=seg["annotated_text"],
        kokoro_voice=voice_info["kokoro_voice"],
        fish_preset=voice_info["fish_preset"],
        tags=seg["tags_applied"],
    ))

# Sort by segment index
tts_requests.sort(key=lambda r: r.seg_index)

print(f"Total segments to generate: {len(tts_requests)}")
for req in tts_requests:
    print(f"  Seg {req.seg_index}: {req.seg_type:12s} | "
          f"{req.kokoro_voice:15s} | {req.speaker}")

Total segments to generate: 14
  Seg 0: narration    | bf_isabella     | Narrateur
  Seg 1: narration    | bf_isabella     | Narrateur
  Seg 2: dialogue     | bf_isabella     | Narrateur
  Seg 3: dialogue     | bm_george       | Cornudet
  Seg 4: narration    | bf_isabella     | Narrateur
  Seg 5: dialogue     | af_sky          | Elisabeth Rousset
  Seg 6: dialogue     | af_sarah        | Comtesse de Breville
  Seg 7: dialogue     | bf_isabella     | Narrateur
  Seg 8: description  | bf_isabella     | Narrateur
  Seg 9: dialogue     | bf_isabella     | Narrateur
  Seg 10: dialogue     | bf_isabella     | Narrateur
  Seg 11: narration    | bf_isabella     | Narrateur
  Seg 12: dialogue     | am_michael      | Comte Hubert de Breville
  Seg 13: dialogue     | af_sky          | Elisabeth Rousset


## Generation audio

Chaque segment est envoye a Kokoro TTS avec la voix correspondante. Les tags expressifs sont conserves dans le texte brut (Kokoro les ignore). Les fichiers audio sont sauvegardes en MP3 (format natif de Kokoro).

In [5]:
import struct

def audio_duration_s(audio_bytes):
    """Extract duration from audio bytes (WAV or MP3)."""
    if len(audio_bytes) < 12:
        return 0.0
    # WAV format (RIFF header)
    if audio_bytes[:4] == b'RIFF':
        channels = struct.unpack_from('<H', audio_bytes, 22)[0]
        sample_rate = struct.unpack_from('<I', audio_bytes, 24)[0]
        bits_per_sample = struct.unpack_from('<H', audio_bytes, 34)[0]
        data_size = struct.unpack_from('<I', audio_bytes, 40)[0]
        bytes_per_sample = bits_per_sample * channels // 8
        if bytes_per_sample == 0:
            return 0.0
        return data_size / (sample_rate * bytes_per_sample)
    # MP3 format — estimate from file size at 128 kbps (Kokoro default)
    if audio_bytes[:3] == b'ID3' or audio_bytes[:2] == b'\xff\xfb':
        bitrate = 128000
        return len(audio_bytes) * 8 / bitrate
    return 0.0

generated = []
t_start = time.time()

for req in tts_requests:
    # Use original text for Kokoro (strip tags for cleaner output)
    clean_text = req.text
    output_name = f"seg_{req.seg_index:02d}_{req.kokoro_voice}.mp3"
    output_path = TTS_OUTPUT_DIR / output_name

    print(f"Generating seg {req.seg_index}: {req.kokoro_voice}...", end=" ")
    audio_data = kokoro_tts(clean_text, req.kokoro_voice)

    if audio_data:
        with open(output_path, "wb") as f:
            f.write(audio_data)
        req.output_file = str(output_path)
        req.duration_s = audio_duration_s(audio_data)
        req.file_size_kb = len(audio_data) / 1024
        req.status = "success"
        print(f"OK ({req.duration_s:.1f}s, {req.file_size_kb:.0f}KB)")
    else:
        req.status = "failed"
        print("FAILED")

    generated.append(req)
    time.sleep(0.2)  # Rate limit

elapsed = time.time() - t_start
success_count = sum(1 for r in generated if r.status == "success")
print(f"\nGenerated {success_count}/{len(generated)} segments in {elapsed:.1f}s")

Generating seg 0: bf_isabella... 

OK (9.0s, 140KB)
Generating seg 1: bf_isabella... 

OK (6.6s, 102KB)
Generating seg 2: bf_isabella... 

OK (6.8s, 107KB)
Generating seg 3: bm_george... 

OK (5.5s, 87KB)
Generating seg 4: bf_isabella... 

...

OK (4.6s, 72KB)
Generating seg 10: bf_isabella... 

OK (6.2s, 97KB)
Generating seg 11: bf_isabella... 

OK (14.7s, 229KB)
Generating seg 12: am_michael... 

OK (8.8s, 137KB)
Generating seg 13: af_sky... 

OK (7.8s, 122KB)

Generated 14/14 segments in 39.6s


## Resultats de la generation

Tableau recapitulatif de tous les segments generes.

In [6]:
print(f"{'Seg':>4} | {'Type':12s} | {'Voice':15s} | {'Speaker':25s} | "
      f"{'Duration':>8s} | {'Size':>7s} | {'Status':>7s}")
print("-" * 100)
total_duration = 0.0
for req in generated:
    dur = f"{req.duration_s:.1f}s" if req.status == "success" else "N/A"
    size = f"{req.file_size_kb:.0f}KB" if req.status == "success" else "N/A"
    print(f"{req.seg_index:>4} | {req.seg_type:12s} | {req.kokoro_voice:15s} | "
          f"{req.speaker:25s} | {dur:>8s} | {size:>7s} | {req.status:>7s}")
    if req.status == "success":
        total_duration += req.duration_s

print(f"\nTotal duration: {total_duration:.1f}s ({total_duration/60:.1f}min)")
print(f"Output directory: {TTS_OUTPUT_DIR.resolve()}")

 Seg | Type         | Voice           | Speaker                   | Duration |    Size |  Status
----------------------------------------------------------------------------------------------------
   0 | narration    | bf_isabella     | Narrateur                 |     9.0s |   140KB | success
   1 | narration    | bf_isabella     | Narrateur                 |     6.6s |   102KB | success
   2 | dialogue     | bf_isabella     | Narrateur                 |     6.8s |   107KB | success
   3 | dialogue     | bm_george       | Cornudet                  |     5.5s |    87KB | success
   4 | narration    | bf_isabella     | Narrateur                 |    19.1s |   299KB | success
   5 | dialogue     | af_sky          | Elisabeth Rousset         |     6.4s |   100KB | success
   6 | dialogue     | af_sarah        | Comtesse de Breville      |     5.8s |    90KB | success
   7 | dialogue     | bf_isabella     | Narrateur                 |     5.5s |    85KB | success
   8 | description  | bf_i

## Export des metadonnees

Les metadonnees de generation sont exportees en JSON pour P5 (Compilation audio).

In [7]:
tts_metadata = {
    "epic": "1028",
    "pass": "P4",
    "description": "TTS generation via Kokoro for audiobook demo",
    "source_text": "Boule de Suif - Guy de Maupassant",
    "tts_model": "kokoro",
    "tts_service_url": KOKORO_URL,
    "stats": {
        "total_segments": len(generated),
        "successful": sum(1 for r in generated if r.status == "success"),
        "failed": sum(1 for r in generated if r.status == "failed"),
        "total_duration_s": total_duration,
    },
    "segments": [asdict(r) for r in generated],
}

meta_path = TTS_OUTPUT_DIR / "tts_generation_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(tts_metadata, f, ensure_ascii=False, indent=2)

print(f"Metadata saved: {meta_path}")
print(f"Segments: {tts_metadata['stats']['successful']}/{tts_metadata['stats']['total_segments']}")
print(f"Total audio: {total_duration:.1f}s")

Metadata saved: tts_output\tts_generation_metadata.json
Segments: 14/14
Total audio: 122.5s


## Analyse de l'utilisation des voix

Repartition des segments par voix et par type.

In [8]:
from collections import Counter

voice_usage = Counter(r.kokoro_voice for r in generated if r.status == "success")
type_usage = Counter(r.seg_type for r in generated if r.status == "success")

print("Voice usage:")
for voice, count in voice_usage.most_common():
    dur = sum(r.duration_s for r in generated if r.kokoro_voice == voice and r.status == "success")
    print(f"  {voice:20s}: {count} segments, {dur:.1f}s")

print(f"\nSegment types:")
for stype, count in type_usage.most_common():
    print(f"  {stype:12s}: {count}")

Voice usage:
  bf_isabella         : 9 segments, 88.2s
  af_sky              : 2 segments, 14.2s
  bm_george           : 1 segments, 5.5s
  af_sarah            : 1 segments, 5.8s
  am_michael          : 1 segments, 8.8s

Segment types:
  dialogue    : 9
  narration   : 4
  description : 1


## Conclusion P4

Les fichiers audio WAV ont ete generes pour chaque segment du texte via Kokoro TTS. Les metadonnees (durees, voix, paths) sont exportees pour la compilation finale P5.

**Production** : Pour une qualite superieure, les memes segments seront generes via FishAudio S2-Pro avec les tags expressifs complets (`[whisper]`, `[shout]`, `[cold]`, etc.).

**P5 (Compilation audio)** : Concatenation des segments avec silences inter-segments, normalisation du volume, et export en fichier audiobook final.